In [ ]:
import pandas as pd
import numpy as np
import kagglehub
import random
import torch
import torch.nn as nn
import os
import re

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score

from transformers import AutoTokenizer, AutoModel, AutoConfig
from torch.utils.data import Dataset, DataLoader
from transformers.utils import logging

import seaborn as sns

In [17]:
class cfg:
    competition = 'learning-agency-lab-automated-essay-scoring-2'
    checkpoint = 'microsoft/deberta-v3-small'
    tokenizer = None
    max_length = 512
    batch_size = 16
    epochs = 10
    learning_rate = 5e-5
    weight_decay = 0.01
    task = 'regression'

In [18]:
#Quadratic Weighted Kappa score
def get_score(y_true, y_pred):
    return cohen_kappa_score(
        y_true,
        y_pred,
        weights="quadratic"
    )

In [19]:
#Sets random to 42
def seed_everything(seed: int=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    
seed_everything(42)

In [20]:
path = kagglehub.competition_download(cfg.competition)

In [21]:
train_df = pd.read_csv(f'{path}/train.csv')

In [22]:
#Label y starting from 0 so classifier works
if train_df['score'].min() == 1:
    train_df['score'] = train_df['score'] - 1

In [ ]:
train_df['word_count'] = train_df['full_text'].str.split(' ').str.len()

train_df['num_sentences'] = train_df['full_text'].apply(lambda x: len([s for s in re.split(r'[.!?]+', str(x)) if s.strip()]))

train_df['sentence_length'] = train_df[['num_sentences', 'word_count']].apply(lambda row: row['word_count']/row['num_sentences'], axis=1)

train_df['num_para'] = train_df['full_text'].apply(lambda x: len([p for p in str(x).split('\n') if p.strip()]))

train_df['para_length'] = train_df [['num_para', 'word_count']].apply(lambda row: row['word_count']/row['num_para'], axis=1)

train_df['long_essay'] = (train_df['word_count']>2500).astype(int)p

In [23]:
#Cross Validation
train_df['fold'] = -1
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, valid_idx) in enumerate(skf.split(train_df, train_df['score'])):
    train_df.loc[valid_idx, 'fold'] = fold

In [24]:
tokenizer = AutoTokenizer.from_pretrained(cfg.checkpoint)

In [25]:
class EssayDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=cfg.max_length):
        self.X = data['full_text']
        if 'score' in data.columns:
            self.y = data['score']
        else:
            self.y = None
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        tokens = self.tokenizer(
            text = self.X.iloc[idx],
            max_length=self.max_length,
            truncation = True,
            padding = 'max_length',
            return_tensors = 'pt'
        )
        item = {
            'input_ids': tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
        }
        if self.y is not None:
            item['labels'] = torch.tensor(self.y.iloc[idx], dtype=torch.float)
        return item


In [26]:
#MeanPooling code taken from online resources
class MeanPooling(nn.Module):
    def __init__(self):
        super(MeanPooling, self).__init__()

    def forward(self, last_hidden_state, attention_mask):
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
        sum_mask = input_mask_expanded.sum(1)
        sum_mask = torch.clamp(sum_mask, min=1e-9)
        mean_embeddings = sum_embeddings/sum_mask
        return mean_embeddings

In [27]:
class EssayModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = AutoModel.from_pretrained(cfg.checkpoint)
        self.mp = MeanPooling()
        self.classifier = nn.Linear(self.model.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        output = self.model(input_ids = input_ids, attention_mask = attention_mask)
        # #CLS Pooling
        # cls_output = output.last_hidden_state[:, 0, :].float()
        # logits = self.classifier(cls_output)
        #Mean Pooling
        mp_output = self.mp(output.last_hidden_state, attention_mask)
        logits = self.classifier(mp_output)
        return logits


In [28]:
print(torch.backends.mps.is_available())
device = torch.device('mps')
#For CUDA:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

True


In [29]:
#Global Debug variable for testing
DEBUG = True

In [31]:
#Store overall best QWK:
all_oof_preds = []
all_oof_labels = []

for fold in range(1):
    valid_fold = train_df[train_df['fold']==fold] 

    valid_dataset = EssayDataset(valid_fold, tokenizer)
    valid_loader = DataLoader(valid_dataset, batch_size=cfg.batch_size, shuffle=False)

    #Create model object
    logging.set_verbosity_error()
    deberta_model = EssayModel().to(device).float()

    deberta_model.load_state_dict(
        torch.load(f'base-deberta.pth', map_location=device)
    )
    deberta_model.eval()

    with torch.no_grad():
        for batch in valid_loader:
            logits = deberta_model(batch['input_ids'].to(device), batch['attention_mask'].to(device)).squeeze(-1)
            all_oof_preds.extend(logits.cpu().numpy())
            all_oof_labels.extend(batch['labels'].numpy())

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
analysis_df = pd.DataFrame({
    'predictions': all_oof_preds,
    'labels': all_oof_labels
})
analysis_df['full_text'] = train_df['full_text']
analysis_df['word_count'] = train_df['word_count']
analysis_df['error'] = (analysis_df['predictions'] - analysis_df['labels']).abs()
analysis_df = analysis_df.sort_values(by='error', ascending = True)
analysis_df['incorrect'] = analysis_df['error']>0.5
analysis_df['sentence_count'] = train_df['num_sentences']
analysis_df['paragraph_count'] = train_df['num_para']

In [ ]:
worst_preds = analysis_df.iloc[-30:].index.tolist()
print('30 most worst error essay indexes:', worst_preds)

In [ ]:
#Get essays for worst preds
for i in worst_preds:
    print(f'Index: {i} \nScore: {analysis_df.loc[i, "labels"]} \nPrediction: {analysis_df.loc[i, "predictions"]} \nEssay: {train_df.loc[i, "full_text"]}\n\n')
    print('-----------------------------------')

In [ ]:
best_preds = analysis_df.iloc[:30].index.tolist()
print('30 most best error essay indexes:', best_preds)

In [ ]:
#Get essays for best preds
for i in best_preds:
    print(f'Index: {i} \nScore: {analysis_df.loc[i, "labels"]} \nPrediction: {analysis_df.loc[i, "predictions"]} \nEssay: {train_df.loc[i, "full_text"]}\n\n')
    print('-----------------------------------')

In [ ]:
def get_token_length(text):
    return len(tokenizer.encode(text, add_special_tokens=False))
analysis_df['token_length'] = analysis_df['full_text'].apply(get_token_length)

In [ ]:
sns.histplot(data = analysis_df, hue = 'incorrect', x='token_length', kde=True)

In [ ]:
sns.histplot(data = analysis_df[analysis_df['error']>0.5], x='token_length', kde=True).set_title('Incorrect Pred >0.5')

In [ ]:
analysis_df.groupby(pd.cut(analysis_df["sentence_count"], 20)).agg(
    count=("error", "size"),
    mean_abs_error=("error", "mean"),
)